## Load prepared analysis

Import the required packages, locate the repository, and load the spatiotemporal analysis prepared in the previous notebook.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import py4dgeo

repo_dir = Path.cwd().parent if Path.cwd().name == "jupyter" else Path.cwd()

data_dir = repo_dir / "kijkduin"
analysis_file = data_dir / "kijkduin.zip"

if not analysis_file.exists():
    raise FileNotFoundError(
        "Analysis archive not found. Run kalman_01_prepare_analysis.ipynb first."
    )

analysis = py4dgeo.SpatiotemporalAnalysis(str(analysis_file))

print(f"Analysis: {analysis_file}")
print(f"Distances shape: {analysis.distances.shape}")
print(f"Corepoints: {analysis.corepoints.cloud.shape[0]}")

## Method configuration

Define the seed-candidate corepoints and the common parameters used for standard 4D-OBC region growing and Kalman-based seed detection. The analysis can use either all corepoints or a selected range. Kalman filtering itself is computed for all corepoints so that region growing has a complete spatial Kalman signal.

In [ ]:
kalman_cache_path = repo_dir / "results" / "kalman_cache"

use_all_corepoints = False
analysis_range_start = 15000
analysis_range_end = 15999

n_corepoints = analysis.corepoints.cloud.shape[0]

if use_all_corepoints:
    analysis_indices = list(range(n_corepoints))
else:
    if not 1 <= analysis_range_start <= analysis_range_end <= n_corepoints:
        raise ValueError("Selected seed-candidate corepoint range is invalid.")

    analysis_indices = list(
        range(analysis_range_start - 1, analysis_range_end)
    )

region_growing_parameters = {
    "window_width": 14,
    "minperiod": 2,
    "height_threshold": 0.05,
    "neighborhood_radius": 1.0,
    "min_segments": 10,
    "thresholds": [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
}

kalman_parameters = {
    "process_sigma": 0.01,
    "min_sigma_obs": 0.005,
    "z_threshold": 1.96,
    "min_seed_duration": 10,
    "min_seed_magnitude": 0.05,
    "seed_direction": "both",
}

print(f"Corepoints available: {n_corepoints}")
print(f"Seed candidates: {len(analysis_indices)}")
print(f"Kalman cache: {kalman_cache_path}")
print(f"Cache exists: {kalman_cache_path.exists()}")

## Original 4D-OBC baseline

Run the standard py4dgeo region-growing algorithm on the selected seed-candidate corepoints. Temporal smoothing from Notebook 1 is retained.

In [ ]:
analysis.invalidate_results(
    seeds=True,
    objects=True,
    smoothed_distances=False,
)

four_dobc = py4dgeo.RegionGrowingAlgorithm(
    seed_candidates=analysis_indices,
    **region_growing_parameters,
)

objects_4dobc = four_dobc.run(analysis, force=True)
seeds_4dobc = list(analysis.seeds)

print(f"Original 4DOBC seeds: {len(seeds_4dobc)}")
print(f"Original 4DOBC objects: {len(objects_4dobc)}")

## Kalman magnitude segmentation

Run Kalman-based seed detection using the magnitude of the filtered surface-change signal. Spatial support and non-maximum suppression are disabled so that the effect of Kalman magnitude detection can be evaluated directly.

In [ ]:
kf_mag = py4dgeo.KalmanRegionGrowingAlgorithm(
    detection_mode="magnitude",
    kalman_cache_path=kalman_cache_path,
    seed_candidates=analysis_indices,
    use_spatial_support=False,
    use_nms=False,
    **kalman_parameters,
    **region_growing_parameters,
)

objects_kf_mag = kf_mag.run(analysis, force=True)
seeds_kf_mag = list(analysis.seeds)

print(f"KF-Mag seeds: {len(seeds_kf_mag)}")
print(f"KF-Mag objects: {len(objects_kf_mag)}")

## Kalman rate segmentation

Run Kalman-based seed detection using the estimated rate of surface change. Spatial support and non-maximum suppression are disabled to evaluate rate-based detection directly.

In [ ]:
kf_rate = py4dgeo.KalmanRegionGrowingAlgorithm(
    detection_mode="rate",
    kalman_cache_path=kalman_cache_path,
    seed_candidates=analysis_indices,
    use_spatial_support=False,
    use_nms=False,
    **kalman_parameters,
    **region_growing_parameters,
)

objects_kf_rate = kf_rate.run(analysis, force=True)
seeds_kf_rate = list(analysis.seeds)

print(f"KF-Rate seeds: {len(seeds_kf_rate)}")
print(f"KF-Rate objects: {len(objects_kf_rate)}")

## Kalman magnitude segmentation with NMS

Run magnitude-based Kalman seed detection with spatial support and non-maximum suppression (NMS). Spatial support validates locally coherent change, while NMS reduces redundant nearby seed detections before region growing.

In [ ]:
kf_mag_nms = py4dgeo.KalmanRegionGrowingAlgorithm(
    detection_mode="magnitude",
    kalman_cache_path=kalman_cache_path,
    seed_candidates=analysis_indices,
    use_spatial_support=True,
    use_nms=True,
    **kalman_parameters,
    **region_growing_parameters,
)

objects_kf_mag_nms = kf_mag_nms.run(analysis, force=True)
seeds_kf_mag_nms = list(analysis.seeds)

print(f"KF-Mag-NMS seeds: {len(seeds_kf_mag_nms)}")
print(f"KF-Mag-NMS objects: {len(objects_kf_mag_nms)}")

## Kalman rate segmentation with NMS

Run rate-based Kalman seed detection with spatial support and non-maximum suppression (NMS). This retains locally supported rate detections while reducing redundant nearby seeds before region growing.

In [ ]:
kf_rate_nms = py4dgeo.KalmanRegionGrowingAlgorithm(
    detection_mode="rate",
    kalman_cache_path=kalman_cache_path,
    seed_candidates=analysis_indices,
    use_spatial_support=True,
    use_nms=True,
    **kalman_parameters,
    **region_growing_parameters,
)

objects_kf_rate_nms = kf_rate_nms.run(analysis, force=True)
seeds_kf_rate_nms = list(analysis.seeds)

print(f"KF-Rate-NMS seeds: {len(seeds_kf_rate_nms)}")
print(f"KF-Rate-NMS objects: {len(objects_kf_rate_nms)}")

## Method summary

Summarize the number of detected seeds and extracted objects for the baseline and all Kalman-based variants.

In [ ]:
summary = pd.DataFrame(
    {
        "method": ["4DOBC", "KF-Mag", "KF-Rate", "KF-Mag-NMS", "KF-Rate-NMS"],
        "n_seeds": [
            len(seeds_4dobc),
            len(seeds_kf_mag),
            len(seeds_kf_rate),
            len(seeds_kf_mag_nms),
            len(seeds_kf_rate_nms),
        ],
        "n_objects": [
            len(objects_4dobc),
            len(objects_kf_mag),
            len(objects_kf_rate),
            len(objects_kf_mag_nms),
            len(objects_kf_rate_nms),
        ],
    }
)

display(summary)

## Save extraction results

Store compact object records, Kalman seed tables, and the method summary for the evaluation notebook.

In [ ]:
import pickle

evaluation_dir = repo_dir / "results" / "kalman_evaluation"
evaluation_dir.mkdir(parents=True, exist_ok=True)

method_objects = {
    "4DOBC": objects_4dobc,
    "KF-Mag": objects_kf_mag,
    "KF-Rate": objects_kf_rate,
    "KF-Mag-NMS": objects_kf_mag_nms,
    "KF-Rate-NMS": objects_kf_rate_nms,
}

saved_objects = {}

for method_name, objects in method_objects.items():
    saved_objects[method_name] = [
        {
            "object_number": object_number,
            "indices": list(obj.indices),
            "start_epoch": int(obj.start_epoch),
            "end_epoch": int(obj.end_epoch),
            "duration_epochs": int(obj.end_epoch - obj.start_epoch + 1),
            "threshold": float(obj.threshold),
            "seed_corepoint": int(obj.seed.index),
            "seed_start_epoch": int(obj.seed.start_epoch),
            "seed_end_epoch": int(obj.seed.end_epoch),
            "seed_duration_epochs": int(
                obj.seed.end_epoch - obj.seed.start_epoch + 1
            ),
        }
        for object_number, obj in enumerate(objects, start=1)
    ]

with open(evaluation_dir / "extracted_objects.pkl", "wb") as file:
    pickle.dump(saved_objects, file)

seed_tables = {
    "kf_mag_seeds.csv": kf_mag.seed_table,
    "kf_rate_seeds.csv": kf_rate.seed_table,
    "kf_mag_nms_seeds.csv": kf_mag_nms.seed_table,
    "kf_rate_nms_seeds.csv": kf_rate_nms.seed_table,
}

for filename, seed_table in seed_tables.items():
    seed_table.to_csv(evaluation_dir / filename, index=False)

summary.to_csv(evaluation_dir / "method_summary.csv", index=False)

print(f"Evaluation results saved to: {evaluation_dir}")